This script follows the capstone stage two black box optimisation challenge, this is the first attempt for week 1

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import os 
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm

In [69]:
# UCB function 
def acquisition_UCB(beta, mean, stddev, candidates):

    ucb = mean + beta * stddev # compute UCB 
    best_index = np.argmax(ucb) # find the argument x of a(x) that maximises 

    # use index to find best candidate function

    

    return np.round(candidates[best_index],6)

def accquisition_expected_improvement(mu, sigma, f_best, xi=0.0):
    """
    Compute Expected Improvement.

    Parameters
    ----------
    mu : array
        Posterior mean
    sigma : array
        Posterior std
    f_best : float
        Best observed value so far
    xi : float
        Exploration parameter (small positive value)

    Returns
    -------
    EI : array
    """

    sigma = np.maximum(sigma, 1e-12)  # avoid division by zero

    improvement = mu - f_best - xi
    Z = improvement / sigma

    EI = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    EI[sigma < 1e-12] = 0.0

    return EI


def accquisition_prob_improve(mu, sigma, f_best, xi=0.01):
    """
    Probability of Improvement (PI) acquisition function.

    Parameters
    ----------
    mu : array-like
        Posterior mean from GP
    sigma : array-like
        Posterior standard deviation from GP
    f_best : float
        Best observed value so far
    xi : float
        Exploration parameter (encourages exploration)

    Returns
    -------
    PI : array
        Probability of improvement at each point
    """

    mu = np.asarray(mu)
    sigma = np.asarray(sigma)

    # avoid division by zero
    sigma = np.maximum(sigma, 1e-12)

    Z = (mu - f_best - xi) / sigma

    PI = norm.cdf(Z)

    # if sigma ~ 0 → no uncertainty → no improvement possible
    PI[sigma < 1e-12] = 0.0

    return PI


# implement GPR with a RBF kernel 
def GPR_with_RBF_kernel_monte_carlo(rbf_lengthscale,
                        noise_assumption,
                        ins,outs,
                        sample_size,
                        in_cols):
    kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed') # user defined length scale 
    #kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds=(1e-3, 1e3)) # letting GP learn the length scale of the data
    model = GaussianProcessRegressor(kernel = kernel, alpha=noise_assumption)

    # normalise all dimensions 
    
   
    #fit data to GPR model 
    model.fit(ins,outs)



    # generate possible coordinate samples for acquisition function

    rng = np.random.default_rng(seed=42)

    sample_ins = rng.random((sample_size, in_cols))
    #print(sample_ins.shape)

    # above sample assumes the coords are between [0,1)

    post_mean, post_std = model.predict(sample_ins ,return_std=True)

    return post_mean, post_std, sample_ins

def GPR_with_RBF_kernel_grid_search(rbf_lengthscale,
                        noise_assumption,
                        ins,outs,
                        n_points,
                        ):
    
    kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds='fixed') # user defined length scale 
    #kernel = RBF(length_scale=rbf_lengthscale, length_scale_bounds=(1e-3, 1e3)) # letting GP learn the length scale of the data
    model = GaussianProcessRegressor(kernel = kernel, alpha=noise_assumption)

    # normalise all dimensions 
    
   
    #fit data to GPR model 
    model.fit(ins,outs)



    # generate possible coordinate samples for acquisition function

    x1 = np.linspace(0,1,n_points)
    x2 = np.linspace(0,1,n_points)

    X1, X2 = np.meshgrid(x1, x2)

    sample_ins = np.vstack([X1.ravel(), X2.ravel()]).T
    #print(sample_ins.shape)

    # above sample assumes the coords are between [0,1)

    post_mean, post_std = model.predict(sample_ins ,return_std=True)

    return post_mean, post_std, sample_ins


def check_unit_cube(X, tol=1e-12):
    """
    Per-dimension check of whether inputs lie in [0,1].
    """

    X = np.asarray(X)

    mins = np.min(X, axis=0)
    maxs = np.max(X, axis=0)

    for i, (mn, mx) in enumerate(zip(mins, maxs)):
        status = (mn >= -tol) and (mx <= 1 + tol)

        print(f"Dim {i}: min={mn:.6f}, max={mx:.6f} -> "
              f"{'OK' if status else 'OUT OF RANGE'}")

    return np.all((mins >= -tol) & (maxs <= 1 + tol))

def compute_acquisition_consistency(results):
    """
    Compute consistency (max pairwise distance) for each noise level.

    Parameters
    ----------
    results : list of dict
        Each dict contains:
        {'noise': ..., 'UCB': ..., 'EI': ..., 'PI': ...}

    Returns
    -------
    summary : list of dict
        [{'noise': ..., 'consistency': ...}, ...]
    """

    summary = []

    for entry in results:
        x_ucb = np.asarray(entry['UCB'])
        x_ei  = np.asarray(entry['EI'])
        x_pi  = np.asarray(entry['PI'])

        # pairwise distances
        d_ue = np.linalg.norm(x_ucb - x_ei)
        d_up = np.linalg.norm(x_ucb - x_pi)
        d_ep = np.linalg.norm(x_ei - x_pi)

        d_max = max(d_ue, d_up, d_ep)

        summary.append({
            'noise': entry['noise'],
            'consistency': d_max
        })

    return summary

def compute_between_noise_distances(results, key):
    distances = []

    for i in range(len(results) - 1):
        x1 = np.asarray(results[i][key], dtype=float)
        x2 = np.asarray(results[i + 1][key], dtype=float)

        dist = np.linalg.norm(x2 - x1)
        distances.append(dist)

    return np.array(distances)  # ← FIX HERE

def select_noise_from_stability(results, key='UCB', tol=1e-6):
    """
    Select a noise level based on stability of the predicted next point.

    Strategy
    --------
    1. Compute distances between predicted points at consecutive noise levels.
    2. Mark transitions where the prediction does NOT change (distance < tol).
    3. Identify the longest consecutive region of stability.
    4. Choose the midpoint of this region as a representative noise level.

    Parameters
    ----------
    results : list of dict
        Each entry has the form:
        {'noise': ..., 'UCB': ..., 'EI': ..., 'PI': ...}

    key : str
        Which acquisition function to use ('UCB', 'EI', or 'PI').

    tol : float
        Tolerance for deciding if two points are "the same".
        If distance < tol → considered stable.

    Returns
    -------
    dict
        {
            'noise': selected noise value,
            'x_next': predicted next point at that noise,
            'index': index of selected noise in results
        }
    """

    # Step 1: compute distances between consecutive noise levels
    # distances[i] = || x(sigma_{i+1}) - x(sigma_i) ||
    distances = compute_between_noise_distances(results, key)

    # Step 2: convert distances into a boolean stability mask
    # True  → no change in predicted point (stable)
    # False → change occurred (instability / jump)
    stable = distances < tol

    # Step 3: find the longest consecutive "True" segment
    max_len = 0        # length of longest stable segment found so far
    best_start = 0     # starting index of that segment

    current_len = 0    # length of current segment being tracked
    current_start = 0  # start index of current segment

    for i, val in enumerate(stable):

        if val:
            # If entering a new stable segment, record where it starts
            if current_len == 0:
                current_start = i

            # Extend current stable segment
            current_len += 1

            # Update best segment if this one is longer
            if current_len > max_len:
                max_len = current_len
                best_start = current_start

        else:
            # Instability encountered → reset current segment
            current_len = 0

    # Step 4: choose midpoint of the longest stable segment
    # This avoids picking boundary values and gives a representative point
    best_index = best_start + max_len // 2

    # Step 5: return the corresponding noise and predicted next point
    return {
        'noise': results[best_index]['noise'],
        'x_next': results[best_index][key],
        'index': best_index
    }

def get_best_observed_input(ins, outs):
    idx = np.argmax(outs)
    return ins[idx]

def select_best_acquisition_from_candidates(x_ucb, x_ei, x_pi, best_input):
    """
    Select the acquisition function whose predicted point
    is closest to the previously observed best input.
    """

    best_input = np.asarray(best_input, dtype=float)

    d_ucb = np.linalg.norm(np.asarray(x_ucb) - best_input)
    d_ei  = np.linalg.norm(np.asarray(x_ei)  - best_input)
    d_pi  = np.linalg.norm(np.asarray(x_pi)  - best_input)

    distances = {
        "UCB": d_ucb,
        "EI": d_ei,
        "PI": d_pi
    }

    best_key = min(distances, key=distances.get)

    return {
        "selected_acquisition": best_key,
        "selected_point": {
            "UCB": x_ucb,
            "EI": x_ei,
            "PI": x_pi
        }[best_key],
        "distances": distances
    }

Need to take the old data and the new inputs and outputs. Can join them together to get the new data set.


In [17]:
# simply paste new inputs below for the next week. 

new_outs=[np.float64(0.0), np.float64(-0.12343145184283744),
           np.float64(-0.30202375198971326), np.float64(-2.1579661153680045),
             np.float64(2687.686511102482), np.float64(-2.6557494178535874),
               np.float64(0.036344135362478075), np.float64(9.3844260752689)]

for x in range(1,9):
    print("function output:", new_outs[x-1])



new_inputs=[np.array([0.00445 , 0.994344]), 
             np.array([0.00445 , 0.994344]),
             np.array([0.9757  , 0.018176, 0.952237]), 
             np.array([0.508167, 0.489502, 0.333094, 0.431327]), 
             np.array([0.377511, 0.871594, 0.988652, 0.967795]), 
             np.array([0.839227, 0.059535, 0.938613, 0.042403, 0.981669]), 
             np.array([0.014159, 0.967762, 0.632863, 0.010833, 0.49052 , 0.157537]), 
             np.array([0.360178, 0.477812, 0.322229, 0.423164, 0.580747, 0.34964 ,0.464133, 0.496224])]





def join_new_data_to_old(new_outs,new_inputs,data_file,new_file_deposit_path):
    # need to join the new y output 
    for i in range(1,9):
        ins=np.load(f"/Users/luke_dev/Documents/ML_AI_imperial_course_codes/capstone_stage_2/{data_file}/function_{i}/initial_inputs.npy")
        outs=np.load(f"/Users/luke_dev/Documents/ML_AI_imperial_course_codes/capstone_stage_2/{data_file}/function_{i}/initial_outputs.npy")
        
        
        #add row index for concatenation of the new inputs 
        new_ins_concat=np.reshape(new_inputs[i-1],(1,new_inputs[i-1].shape[0])) 
        # make single output into array for concat 
        new_outs_concat=np.array([new_outs[i-1]])
        


        new_input_arr=np.concatenate((ins,new_ins_concat),axis=0)
        print(new_input_arr.shape)
        new_output_arr=np.concatenate((outs,new_outs_concat),axis=0)
        print(new_output_arr.shape)

        print(os.getcwd())
        
        os.mkdir(f'{new_file_deposit_path}/function_{i}')
        np.save( f'{new_file_deposit_path}/function_{i}/inputs.npy',new_input_arr)
        np.save( f'{new_file_deposit_path}/function_{i}/outputs.npy',new_output_arr)





data_file='initial_data'  
new_file_deposit_path='week_2_data'
join_new_data_to_old(new_outs,new_inputs,data_file,new_file_deposit_path)



function output: 0.0
function output: -0.12343145184283744
function output: -0.30202375198971326
function output: -2.1579661153680045
function output: 2687.686511102482
function output: -2.6557494178535874
function output: 0.036344135362478075
function output: 9.3844260752689
(11, 2)
(11,)
/Users/luke_dev/Documents/ML_AI_imperial_course_codes/capstone_stage_2/week_2


FileExistsError: [Errno 17] File exists: 'week_2_data/function_1'

In [ ]:
for x in range(1,9):
    print(f"function {x} output:", new_outs[x-1])

# core function loop , though I dont think we can do this anymore
def bayesian_optimisation_loop():
    for i in range(1,9):
        ins=np.load(f"/Users/luke_dev/Documents/ML_AI_imperial_course_codes/capstone_stage_2/week_2/week_2_data/function_{i}/inputs.npy")
        outs=np.load(f"/Users/luke_dev/Documents/ML_AI_imperial_course_codes/capstone_stage_2/week_2/week_2_data/function_{i}/outputs.npy")
        # print("shape outputs",np.shape(outs))
        # print("shape inputs",np.shape(ins))
        # print(np.min(ins),np.max(ins))
        # print(np.min(outs),np.max(outs))
   
        #Parameters of the problem. Feel free to change them and play around with
        noise_assumption = 1e-10 #Noise assumption, a hyperparameter
        rbf_lengthscale = 0.5 #Length scale parameter
        sample_size=100000
        in_cols=np.shape(ins)[1]
        Beta=5 

        check_unit_cube(ins) # checking inputs lie in [0,1]^{dimensions}

        post_mean, post_std,sample_ins=GPR_with_RBF_kernel(rbf_lengthscale,
                                noise_assumption,
                                ins,outs,
                                sample_size,
                                in_cols)

        x_next=acquisition_UCB(Beta,post_mean, post_std,sample_ins)
        print("next input values",x_next)


# 2-D input to 1-D out is hard to visualise clearly so wont bother

bayesian_optimisation_loop()


NameError: name 'new_outs' is not defined

In [76]:
# optimisation loop

for k in range(1,9):
    ins=np.load(f"/Users/luke_dev/Documents/ML_AI_imperial_course_codes/capstone_stage_2/week_2/week_2_data/function_{k}/inputs.npy")
    #print(ins)
    outs=np.load(f"/Users/luke_dev/Documents/ML_AI_imperial_course_codes/capstone_stage_2/week_2/week_2_data/function_{k}/outputs.npy")
   # print(outs)

    # going to use a highly refined grid since the input space is only 2-d so should be easy enough to search 



    #real_noise_std = 1e-10 #Real noise needs to be positive for code to work, instead of zero set 1e-10

    noise_assumption_list=np.logspace(-10,-1) #Noise assumption, a hyperparameter
    rbf_lengthscale = 0.5 #Length scale parameter
    n_points=int(1e3) 
    Beta=2 # exploration term for UCB
    Xi=0.1 # exploration term for PI 

    sample_size=int(1e6)
    in_cols=ins.shape[1]

    # def noise_assumption_sweep_grid_search(ins,outs,noise_assumption_list,rbf_lengthscale,n_points,Beta,Xi):
        
    #     results = []



    #     for i in noise_assumption_list:
    #         noise_assumption = i
    #         post_mean, post_std,sample_ins=GPR_with_RBF_kernel_grid_search(rbf_lengthscale,
    #                                 noise_assumption,
    #                                 ins,outs,
    #                                 n_points,
    #                                 )

        

    #         x_next_UCB=acquisition_UCB(Beta,post_mean, post_std,sample_ins)

            
    #         #X_next_UCB_list.append(x_next_UCB)

    #         f_best = np.max(outs)

    #         ei = accquisition_expected_improvement(post_mean, post_std, f_best)

    #         x_next_EI = sample_ins[np.argmax(ei)]

    #         #X_next_EI_list.append(x_next_EI)

        

    #         pi = accquisition_prob_improve(post_mean, post_std, f_best, xi=Xi)

    #         x_next_PI = sample_ins[np.argmax(pi)]

    #         #X_next_PI_list.append(x_next_PI)


    #         results.append({
    #                         "noise": i,
    #                         "UCB": x_next_UCB,
    #                         "EI": x_next_EI,
    #                         "PI": x_next_PI
    #                     })
    #         # print("noise assumed:",i)
    #         # print("next input values according to UCB",x_next_UCB)
    #         # print("next input values according to EI",x_next_EI)
    #         # print("next input values according to PI",x_next_PI)

    #     #return X_next_EI_list,X_next_PI_list,X_next_UCB_list
    #     return results


    def noise_assumption_sweep_monte_carlo(ins,outs,sample_size,in_cols,rbf_lengthscale, noise_assumption_list,Beta,Xi):
        

        results=[]

        for i in noise_assumption_list:
            noise_assumption = i
            post_mean, post_std,sample_ins=GPR_with_RBF_kernel_monte_carlo(rbf_lengthscale,
                                noise_assumption,
                                ins,outs,
                                sample_size,
                                in_cols)
            
    


            x_next_UCB=acquisition_UCB(Beta,post_mean, post_std,sample_ins)

            
            #X_next_UCB_list.append(x_next_UCB)

            f_best = np.max(outs)

            ei = accquisition_expected_improvement(post_mean, post_std, f_best)

            x_next_EI = sample_ins[np.argmax(ei)]
            x_next_EI_std=post_std[np.argmax(ei)]

            #X_next_EI_list.append(x_next_EI)

        

            pi = accquisition_prob_improve(post_mean, post_std, f_best, xi=Xi)

            x_next_PI = sample_ins[np.argmax(pi)]

            #X_next_PI_list.append(x_next_PI)



            # print("noise assumed:",i)
            # print("next input values according to UCB",x_next_UCB)
            # print("next input values according to EI",x_next_EI)
            # print("next input values according to PI",x_next_PI)
            results.append({
                            "noise": i,
                            "UCB": x_next_UCB,
                            "EI": x_next_EI,
                            "PI": x_next_PI
                        })

        #return X_next_EI_list,X_next_PI_list,X_next_UCB_list
        return results 

    #X_next_EI_list_GS,X_next_PI_list_GS,X_next_UCB_list_GS=noise_assumption_sweep_grid_search(ins,outs,noise_assumption_list,rbf_lengthscale,n_points,Beta,Xi) 
    #results_GS=noise_assumption_sweep_grid_search(ins,outs,noise_assumption_list,rbf_lengthscale,n_points,Beta,Xi) 

    # can also do the monte carlo sampling technique 
    results_MC=noise_assumption_sweep_monte_carlo(ins,outs,sample_size,in_cols,rbf_lengthscale, noise_assumption_list,Beta,Xi)
    #X_next_EI_list_MC,X_next_PI_list_MC,X_next_UCB_list_MC = noise_assumption_sweep_monte_carlo(ins,outs,sample_size,in_cols,rbf_lengthscale, noise_assumption_list,Beta,Xi)


    # now going to look at the distance clustering across the 3 accquisition functions and then the grid search monte carlo

    # consistency_results = compute_acquisition_consistency(results_GS)

    # for row in consistency_results:
    #     print(row)

    # consistency_results = compute_acquisition_consistency(results_MC)

    # for row in consistency_results:
    #     print(row)
    
    
    # ucb_noise_distances = compute_between_noise_distances(results_MC, 'UCB')
    # ei_noise_distances  = compute_between_noise_distances(results_MC, 'EI')
    # pi_noise_distances  = compute_between_noise_distances(results_MC, 'PI')

    # for row in ucb_noise_distances:
    #     print(row)

    print(f"function number {k}")

    # get best observed input
    best_input = get_best_observed_input(ins, outs)

    # get stable noise + next point for each acquisition
    best_ucb = select_noise_from_stability(results_MC, key='UCB')
    best_ei  = select_noise_from_stability(results_MC, key='EI')
    best_pi  = select_noise_from_stability(results_MC, key='PI')

    x_ucb = best_ucb['x_next']
    x_ei  = best_ei['x_next']
    x_pi  = best_pi['x_next']

    print("Next point UCB:", x_ucb)
    print("Next point EI :", x_ei)
    print("Next point PI :", x_pi)

    # select best acquisition based on proximity to known optimum
    final_choice = select_best_acquisition_from_candidates(x_ucb, x_ei, x_pi, best_input)

    print("\nBest observed input:", best_input)
    print("Distances to best:", final_choice["distances"])
    print("Selected acquisition:", final_choice["selected_acquisition"])
    if k==1:

       print("Selected next point:",
      ", ".join(f"{xi:.7f}" for xi in final_choice["selected_point"]))
       
    else:
        print("Selected next point:",
      ", ".join(f"{xi:.6f}" for xi in final_choice["selected_point"]))





        






function number 1
Next point UCB: [9.99235e-01 2.98000e-04]
Next point EI : [9.99234532e-01 2.98424546e-04]
Next point PI : [9.98971807e-01 1.17545877e-04]

Best observed input: [0.73102363 0.73299988]
Distances to best: {'UCB': np.float64(0.7802495614780308), 'EI': np.float64(0.7802490020016528), 'PI': np.float64(0.780328607262498)}
Selected acquisition: EI
Selected next point: 0.9992345, 0.0002984
function number 2
Next point UCB: [0.99995  0.261005]
Next point EI : [0.99997255 0.99984253]
Next point PI : [0.74476216 0.96750973]

Best observed input: [0.70263656 0.9265642 ]
Distances to best: {'UCB': np.float64(0.7289474115949177), 'EI': np.float64(0.3062325987024426), 'PI': np.float64(0.05874608852533181)}
Selected acquisition: PI
Selected next point: 0.744762, 0.967510
function number 3
Next point UCB: [0.957464 0.99764  0.999489]
Next point EI : [0.98668696 0.99111293 0.99401522]
Next point PI : [5.30729809e-01 9.39832478e-01 2.23845581e-04]

Best observed input: [0.49258141 0.611